### DTO, Entity
- 실무에서는 비밀번호 숨기기보다 데이터 가공(Data Transformation) 목적으로 DTO를 훨씬 많이 사용

In [ ]:
'''
1. Entity 생성
2. Entity 출력
3. DTO 생성
4. 비밀번호 제거
5. 날짜 형식 변환
6. 회원 등급 변환
7. 전화번호 마스킹
8. DTO 최종 결과 확인
'''

'''
SQLite
 ↓
Entity
 ↓
DTO
 ↓
사용자 응답
'''

In [4]:
from dataclasses import dataclass, asdict
from datetime import datetime
import sqlite3

'''
id
name
email
password_hash
phone
grade
created_at
'''

'''
고객명: 홍길동
이메일: hong@test.com
비밀번호: x7a9bc2de4f
전화번호: 01012345678
회원등급: 3
가입일: 2026-06-03
'''

'\n고객명: 홍길동\n이메일: hong@test.com\n비밀번호: x7a9bc2de4f\n전화번호: 01012345678\n회원등급: 3\n가입일: 2026-06-03\n'

In [5]:
# SQLite DB 생성

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    password_hash TEXT NOT NULL,
    phone TEXT NOT NULL,
    grade INTEGER NOT NULL,
    created_at TEXT NOT NULL
)
""")

conn.commit()
conn.close()

print("DB 생성 완료")

DB 생성 완료


In [6]:
# 실제 고객 데이터 저장

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute(
    """
    INSERT INTO customers(
        name,
        email,
        password_hash,
        phone,
        grade,
        created_at
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    (
        "홍길동",
        "hong@test.com",
        "x7a9bc2de4f",
        "01012345678",
        3,
        "2026-06-03 14:30"
    )
)

conn.commit()
conn.close()

print("고객 저장 완료")

고객 저장 완료


In [7]:
# DB 확인

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM customers
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

# 여기까지 고객 조회 결과가 단순한 튜플의 형태로 출력된다.
# 실무에서는 이렇게 사용할 수 없다. (260603)

(1, '홍길동', 'hong@test.com', 'x7a9bc2de4f', '01012345678', 3, '2026-06-03 14:30')


In [8]:
# Customer Entity 생성

@dataclass
class CustomerEntity:
    """
    DB의 customers 테이블을 표현하는 Entity
    Entity는 DB의 모든 정보를 가지고 있다.
    """

    id: int
    name: str
    email: str

    # 사용자 비밀번호 해시
    password_hash: str

    # 휴대폰 번호
    phone: str

    # 회원 등급
    # 1 = Bronze
    # 2 = Silver
    # 3 = Gold
    grade: int

    # 가입일
    created_at: datetime

In [ ]:
# DB -> Entity 변환

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM customers
LIMIT 1
""")

row = cursor.fetchone()

conn.close()

customer = CustomerEntity(
    id=row[0],
    name=row[1],
    email=row[2],
    password_hash=row[3],
    phone=row[4],
    grade=row[5],
    created_at=row[6]
)

customer

# Customer Entity로 저장되어 출력 (260603)

CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f', phone='01012345678', grade=3, created_at='2026-06-03 14:30')

In [ ]:
# 고객에게 전달할 객체를 정의

@dataclass
class CustomerResponseDTO:

    id: int
    name: str
    email: str
    phone: str
    grade: str
    created_at: str

    @classmethod
    def from_entity(cls, customer):

        grade_map = {
            1: "Bronze",
            2: "Silver",
            3: "Gold"
        }

        masked_phone = (
            customer.phone[:3] # 세번째 자리까지 포함
            + "****"           # 가운데 숫자는 마스킹 처리
            + customer.phone[-4:]  # 끝에서 네번째 자리부터 출력
        )

        return cls(             # 클래스도 메서드 처럼 return 값을 정의할 수 있다.
            id=customer.id,
            name=customer.name,
            email=customer.email,
            phone=masked_phone,                  # 클래스 메서드에서 정의된 값으로 일부분을 마스킹하여 전달 (260603)
            grade=grade_map[customer.grade],     # 클래스 메서드에서 정의된 값으로 숫자 -> 명칭으로 변경하여 전달 (260603)
            created_at=customer.created_at
        )

In [ ]:
# Entity -> 고객 DTO 변환

response = CustomerResponseDTO.from_entity(customer)
response 

# 출력 결과 확인,
# grade = Gold
# phone = 가운데 부분 마스킹 처리

CustomerResponseDTO(id=1, name='홍길동', email='hong@test.com', phone='010****5678', grade='Gold', created_at='2026-06-03 14:30')